## Imports

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

## Define Env Variables

In [0]:
%run /Workspace/Users/kavyajeetbora@gmail.com/fmcg_databricks/consolidated_pipeline/dimension_data_processing/00_config

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
# dbutils.widgets.text("catalog", "fmcg", "Catalog")
# dbutils.widgets.text("data_source", "customers")
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

## Bronze Layer Processing

In [0]:
df_bronze = (
    spark.read.format("csv")
    .option("inferSchema", True)
    .option('header', True)
    .load(f"s3://sportsbar-raw-data-5684/{data_source}/*.csv")
    .withColumn('read_timestamp', F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
display(df_bronze.limit(10))

In [0]:
(
    df_bronze.write.format('delta')
    .option("delta.enableChangeDataFeed", True)
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")
)

## Silver

This layer will have clean data
1. deduplicates
2. standardization of columns
...etc etc

In [0]:
df_bronze.groupBy("customer_id").agg(F.count("customer_id").alias("customer_count")).filter(F.col("customer_count")>1).show()

In [0]:
print("Before:",df_bronze.count())
df_silver_raw = df_bronze.dropDuplicates(subset=['customer_id'])
print("After",df_silver_raw.count())

In [0]:
df_silver = (
    df_silver_raw
    .withColumn(
        "customer_name",
        F.trim(F.initcap(F.col("customer_name")))
    )
    .withColumn(
        "customer_id", 
        F.trim(F.col("customer_id").cast("string"))
    )
)

city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}

allowed = ["Bengaluru", "Hyderabad", "New Delhi"]


df_silver = (
    df_silver.replace(city_mapping, subset=['city'])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
        .when(F.col("city").isin(allowed), F.col("city"))
        .otherwise(None)
    )
)


## Updated Null City Mapping: Business Confirmation Note: City corrections confirmed by business team

customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}
df_city_correction = spark.createDataFrame(customer_city_fix.items(), ["fixed_customer_id", "fixed_city"])

## Join the corrected data
df_silver = (
    df_silver.alias("T1")
    .join(
        df_city_correction.alias("T2"),
        F.col("T1.customer_id") == F.col("T2.fixed_customer_id"),
        "left"
    )
    .withColumn(
        "city",
        F.coalesce(F.col("T1.city"), F.col("T2.fixed_city"))
    )
    .drop("fixed_city", "fixed_customer_id")
)

## Standardizing the Columns as per the existing table -> customer_code, customer, market, platform and channel
df_silver = (
    df_silver
    .withColumnRenamed("customer_id", "customer_code")
    .withColumn(
        "customer",
        F.concat_ws("-", F.col("customer_name"), F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    .withColumn("market", F.lit('India'))
    .withColumn("platform", F.lit('Sports Bar'))
    .withColumn("channel", F.lit('Aquisition'))
    .drop('customer_name')
)

In [0]:
display(df_silver)

In [0]:
(
    df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .option('mergeSchema', True)
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")
)

## Gold

In [0]:
df_gold = spark.read.format('delta').table(f"{catalog}.{silver_schema}.{data_source}")

(
    df_gold.write.format('delta')
    .option('enableChangeDataFeed', True)
    .mode('overwrite')
    .option('mergeSchema', True)
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")
)


Upsert the data to the master `dim_customer` table: 
- When ids match -> Update
- When ids not matched -> Insert

In [0]:
dim_customer = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.dim_{data_source}")
df_gold = spark.read.format('delta').table(f"{catalog}.{silver_schema}.{data_source}")

dim_customer.alias("target").merge(
    source=df_gold.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()